# Qwen3-0.6B × FlashInfer-backed AttentionBackend (educational)

** 자매 프로젝트**: `../ptr/vllm_unified/` — 같은 4단 구조, 직접 작성한 Triton 커널.

이 노트북은 그 `vllm_unified` 와 **같은 조직**(14셀) 을 유지하며, 다른 것은 두 곳뿐:

- **cell 7**: `triton_attention_unified` smoke test → **CSR 변환 데모** (`_build_paged_csr`)
- **cell 10**: LLM 로드 시 플러그인이 같은 `CUSTOM` 슬롯에 이번엔 `MyFlashInferBackend` 를 올림

나머지 — plugin entry point, KV cache shape, observability 로그 관찰 포인트 — 는 동일.

**이 프로젝트의 이야기**: vllm_unified 는 커널 1개로 prefill/decode/chunked 를 통합하는데, FlashInfer 는 wrapper 가 prefill / decode 가 나뉘어 있어서 그 레벨의 수학적 통합이 불가능하다. 따라서 이 백엔드는 vllm_multiseq 의 split-dispatch 단계에 멈춰 있다. 대신 얻는 것은 vLLM v1 이 외부 라이브러리를 어떻게 '갈아끼우는지' 의 구체적 모습이다.

## 준비물 & 설치

- CUDA GPU 필수 (Hopper/Ampere/Blackwell). FlashInfer 는 SM 7.5+ 지원.
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)
- flashinfer-python >= 0.2.0

```bash
# 노트북 디렉토리에서
pip install -e .
# 자매 프로펠트 와 같은 venv 에 둘 이상 설치 금지 — CUSTOM 슬롯 이 상호 충돌.
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = flashinfer_attention_backend:register`) 가 vLLM 프로세스 (메인·엔진 코어·워커) 에서 자동으로 `register()` 를 호출해 준다. 즉 **노트북 안에서 `register()` 를 수동 호출하지 않는다**.

절차:
1. cell 3 — 환경 확인 (cuda + vllm + flashinfer)
2. cell 8 — CSR 변환 데모 (`_build_paged_csr`)
3. cell 11 — LLM 생성 (plugin 이 이미 CUSTOM 슬롯 점유)
4. cell 12~14 — generate + 실행 증거 확인

In [1]:
import torch, vllm
import flashinfer

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('flashinfer:', getattr(flashinfer, '__version__', 'unknown'))

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

# 이 노트북은 vLLM 0.19.x 기준으로 작성됨. 다른 버전에서는 내부 API drift 가능.
assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)

cuda: True
gpu: NVIDIA GeForce RTX 5090
vllm: 0.19.1
flashinfer: 0.6.6


## Qwen3-0.6B 구조 요약

| 항목 | 값 | FlashInfer 제약 충족 여부 |
|---|---|---|
| hidden_size | 1024 | — |
| Q heads | 16 | — |
| KV heads | 8 (GQA 2:1) | OK (`num_kv_heads` 인자로 전달) |
| head_dim | 128 | OK (FlashInfer 허용: 64/128/256) |
| layers | 28 | — |

vLLM `block_size` 기본값 16 도 FlashInfer 허용 집합 {1, 16, 32, 64} 에 포함.

In [2]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

assert hd in (64, 128, 256), 'FlashInfer 는 head_dim ∈ {64,128,256} 만 허용'

/home/osehn/orchestrate/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


hidden: 1024
Q heads: 16 / KV heads: 8
head_dim: 128
layers: 28


## FlashInfer 백엔드의 역할 — split dispatch

vllm_unified 와 달리, FlashInfer wrapper 는 prefill / decode 이 **클래스 자체로 분리**되어 있다:

- `BatchPrefillWithPagedKVCacheWrapper` — q_len > 1 용, `qo_indptr` 필수
- `BatchDecodeWithPagedKVCacheWrapper` — q_len == 1 용, `qo_indptr` 없음

따라서 MetadataBuilder 가 매 forward 마다 요청 그룹을 둘로 나눠야 한다:

| Batch 형태 | q_len, s_len 관계 | 그룹 |
|---|---|---|
| Pure prefill | q_len == s_len | prefill |
| Decode | q_len == 1 | decode |
| Chunked prefill | 1 < q_len < s_len | **prefill** 에 합류 (q_len > 1 이므로) |
| Mixed | 위의 조합 | 두 그룹으로 가르며 쓠다 |

즉 **chunked 는 prefill wrapper 가 처리**한다. q_len > 1 이면 모두 prefill 그룹.

FlashInfer 가 원하는 메타데이터 형식 (CSR):

```
kv_indptr        : (B+1,) int32 — 누적 페이지 수
kv_indices       : (sum_pages,) int32 — flat page ids
kv_last_page_len : (B,) int32 — 마지막 페이지 유효 길이 (1..page_size)
qo_indptr        : (B+1,) int32 — prefill 만, query 토큰 오프셋
```

다음 셀은 이 변환을 시각적으로 확인한다.

In [3]:
from flashinfer_attention_backend import _build_paged_csr

help(_build_paged_csr)

Help on function _build_paged_csr in module flashinfer_attention_backend:

_build_paged_csr(block_table: 'torch.Tensor', seq_lens: 'torch.Tensor', page_size: 'int') -> 'tuple[torch.Tensor, torch.Tensor, torch.Tensor]'
    Convert vLLM's (block_table, seq_lens) into FlashInfer's CSR triple.

    Returns (kv_indptr, kv_indices, kv_last_page_len), all int32, all on
    block_table's device. FlashInfer requires int32 here — int64 silently
    misindexes.

    Educational implementation: a Python list-comprehension. Production
    vLLM does this on the GPU via cumsum + gather.



In [4]:
# CSR 변환 데모: 3개 시퀀스, page_size=16
# seq 0: 길이 44 → 3페이지, 마지막 12
# seq 1: 길이 21 → 2페이지, 마지막 5
# seq 2: 길이 16 → 1페이지, 마지막 16
import torch

page_size = 16
block_table = torch.tensor([
    [3, 7, 11,  0,  0],   # seq 0 의 페이지 ids (뒤 0은 미사용)
    [4, 8,  0,  0,  0],   # seq 1
    [9, 0,  0,  0,  0],   # seq 2
], dtype=torch.int32, device='cuda')
seq_lens = torch.tensor([44, 21, 16], dtype=torch.int32, device='cuda')

kv_indptr, kv_indices, kv_last_page_len = _build_paged_csr(block_table, seq_lens, page_size)

print('kv_indptr        =', kv_indptr.tolist())
print('kv_indices       =', kv_indices.tolist())
print('kv_last_page_len =', kv_last_page_len.tolist())

# 수동 검증
assert kv_indptr.tolist() == [0, 3, 5, 6], 'kv_indptr 변환 오류'
assert kv_indices.tolist() == [3, 7, 11, 4, 8, 9], 'kv_indices 변환 오류'
assert kv_last_page_len.tolist() == [12, 5, 16], 'kv_last_page_len 변환 오류'
assert kv_indptr.dtype == torch.int32, 'FlashInfer 는 int32 필수 (int64 는 조용히 misindex)'
print()
print('CSR 변환 PASSED — 이게 매 forward 마다 builder 에서 일어난다.')

kv_indptr        = [0, 3, 5, 6]
kv_indices       = [3, 7, 11, 4, 8, 9]
kv_last_page_len = [12, 5, 16]

CSR 변환 PASSED — 이게 매 forward 마다 builder 에서 일어난다.


## Plugin + FlashInfer wrapper 연결

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_flashinfer_backend = "flashinfer_attention_backend:register"
```

**Backend 의 split-dispatch 로직** (`MyFlashInferImpl.forward`):
```python
if attn_metadata.prefill_count > 0:
    self._prefill_wrapper.plan(qo_indptr=..., paged_kv_indptr=..., ...)
    o_pre = self._prefill_wrapper.run(q_pre, paged_kv)
    output.index_copy_(0, prefill_token_slice, o_pre)

if attn_metadata.decode_count > 0:
    self._decode_wrapper.plan(indptr=..., indices=..., ...)
    o_dec = self._decode_wrapper.run(q_dec, paged_kv)
    output.index_copy_(0, decode_token_slice, o_dec)
```

vllm_unified 의 `output[:N].copy_(o_flat)` 한 줄과 비교해서 보면 이 프로젝트의 재번의윈 차이가 정확히 이 두 분기에 있다. 다음은 split-dispatch 로 인해 추가되는 launch 가 쓰니는지 관찰:
- `FIRE_COUNTER["prefill_calls"]` / `FIRE_COUNTER["decode_calls"]` 가 따로 올라간다.

In [5]:
# entry point 등록 상태 확인
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_flashinfer_backend']
assert mine, '`my_flashinfer_backend` entry point 가 보이지 않는다. pip install -e . 확인.'
print('plugin registered:', mine[0].name, '->', mine[0].value)

plugin registered: my_flashinfer_backend -> flashinfer_attention_backend:register


In [6]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

# vllm_unified 와 동일: max_num_batched_tokens=64 로 chunked prefill trigger.
# 차이 점: chunked 가 와도 prefill_wrapper 그룹으로 합류 (q_len > 1).
llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=4,
    max_model_len=2048,
    max_num_batched_tokens=64,   # ← chunked prefill trigger 포인트
)

INFO 05-07 22:46:00 [utils.py:233] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'max_num_batched_tokens': 64, 'max_num_seqs': 4, 'disable_log_stats': True, 'enforce_eager': True, 'attention_backend': <AttentionBackendEnum.CUSTOM: None>}
INFO 05-07 22:46:02 [model.py:549] Resolved architecture: Qwen3ForCausalLM
WARNING 05-07 22:46:02 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 05-07 22:46:02 [model.py:1678] Using max model len 2048
INFO 05-07 22:46:02 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=64.
INFO 05-07 22:46:02 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 05-07 22:46:02 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-07 22:46:02 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-07 22:46:03 

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.94it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  1.93it/s]
(EngineCore pid=2601369) 


(EngineCore pid=2601369) INFO 05-07 22:46:09 [default_loader.py:384] Loading weights took 0.52 seconds
(EngineCore pid=2601369) INFO 05-07 22:46:10 [gpu_model_runner.py:4820] Model loading took 1.12 GiB memory and 1.683287 seconds
(EngineCore pid=2601369) INFO 05-07 22:46:10 [gpu_worker.py:436] Available KV cache memory: 26.9 GiB
(EngineCore pid=2601369) INFO 05-07 22:46:10 [kv_cache_utils.py:1319] GPU KV cache size: 251,808 tokens
(EngineCore pid=2601369) INFO 05-07 22:46:10 [kv_cache_utils.py:1324] Maximum concurrency for 2,048 tokens per request: 122.95x
(EngineCore pid=2601369) INFO 05-07 22:46:10 [core.py:283] init engine (profile, create kv cache, warmup model) took 0.71 seconds


(EngineCore pid=2601369) 2026-05-07 22:46:10,804 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=2601369) 2026-05-07 22:46:10,809 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends


(EngineCore pid=2601369) INFO 05-07 22:46:13 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=2601369) WARNING 05-07 22:46:13 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=2601369) WARNING 05-07 22:46:13 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=2601369) INFO 05-07 22:46:13 [vllm.py:1025] Cudagraph is disabled under eager mode
(EngineCore pid=2601369) INFO 05-07 22:46:13 [compilation.py:292] Enabled custom fusions: norm_quant, act_quant


In [7]:
# vllm_unified 와 동일한 prompt 셋. 긴 prompt 가 chunk 로 쪼개진다.
long_prompt = 'In the long history of artificial intelligence research, from the early symbolic AI of the 1950s through the neural network revival of the 1980s, the deep learning breakthroughs of the 2010s, and the transformer-based large language models of the 2020s, one theme has remained constant: ' + 'the answer is'

prompts = [
    'The capital of France is',
    long_prompt,
    'Shakespeare wrote the play',
    'Python was created by',
]
out = llm.generate(prompts, SamplingParams(temperature=0, max_tokens=16))
for i, o in enumerate(out):
    print(f'[{i}] (prompt {len(prompts[i])} chars) {o.outputs[0].text[:80]}')

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s](EngineCore pid=2601369) MyFlashInferImpl.forward fired num_seqs=2 (prefill=2 decode=0) max_q_len=59 tokens=64
(EngineCore pid=2601369) MyFlashInferImpl.forward fired num_seqs=4 (prefill=3 decode=1) max_q_len=16 tokens=26
Processed prompts: 100%|██████████| 4/4 [00:00<00:00, 12.49it/s, est. speed input: 279.04 toks/s, output: 200.65 toks/s]

[0] (prompt 24 chars)  Paris. The capital of Italy is Rome. The capital of Spain is Madrid.
[1] (prompt 300 chars)  the key to solving the problem. This is the core of the problem. The
[2] (prompt 26 chars)  "Macbeth" in 1606. The play is based on
[3] (prompt 21 chars)  a group of developers who wanted to create a new way to write code that is


## 검증 — prefill_calls 와 decode_calls 가 모두 와야 한다

엔진 코어 stderr 에 다음 같은 로그가 찍혀있어야:

```
MyFlashInferImpl.forward fired num_seqs=N (prefill=P decode=D) max_q_len=Q tokens=T
```

**관찰 포인트**:
- 첫 forward: 짧은 prompt 의 prefill → `prefill=1, decode=0`
- 긴 prompt 첫 chunk: `q_len=64, s_len=64` → 여전히 prefill (q_len > 1)
- 긴 prompt 두 번째 chunk: `q_len=64, s_len=128` → **chunked**, 하지만 q_len > 1 이므로 여전히 prefill_wrapper 로 감
- 동시에 이미 decode 중인 세션들 → `decode=k`

따라서 **이상적으로는 prefill+decode 두 그룹이 동시에 0이 아닌 forward** 가 최소 한 번 관찰되어야 한다. 하지만 vLLM 스케줄러가 대머지에 좌우되므로 정확하게 공존하는 순간은 prompt 수·길이에 따라 달라진다.

**vllm_unified 와의 관찰 포인트 대조**:

| | vllm_unified | flashinfer (이 프로젝트) |
|---|---|---|
| chunked 가 공존하는 증거 | `chunked=C > 0` | `prefill_calls > 0` (chunked 도 prefill 그룹) |
| prefill+decode 공존 | 같은 forward 에서 세 수치 동시 증가 | 같은 forward 에서 **launch 가 둘** (`prefill_calls += 1, decode_calls += 1`) |
| forward 당 launch 수 | 1 | 최대 2 |

마지막 줄이 이 프로젝트의 핵심 관찰 포인트다 — 같은 일을 1 launch 로 하는 unified 대비, 이 backend 는 외부 라이브러리 제약 때문에 2 launch 로 쓰고 있다.

In [8]:
# 노트북 프로세스에서 확인 가능한 부분
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyFlashInferBackend' in path
print('OK — plugin 자동 등록 확인')
print()
print('FlashInfer 실행 증거 = 위 cell 출력 바로 위의 `MyFlashInferImpl.forward fired` 로그')
print('→ prefill=... decode=... 두 수치가 다 0이 아닌 줄을 찾으면 split-dispatch 로 돌아갔다는 뜻')

CUSTOM slot -> flashinfer_attention_backend.MyFlashInferBackend
OK — plugin 자동 등록 확인

FlashInfer 실행 증거 = 위 cell 출력 바로 위의 `MyFlashInferImpl.forward fired` 로그
→ prefill=... decode=... 두 수치가 다 0이 아닌 줄을 찾으면 split-dispatch 로 돌아갔다는 뜻


## 이 프로젝트의 위치 & 마무리

```
vllm_attn/
  ├─ ptr/vllm_unified/    Triton 커널 1개, prefill+decode+chunked 통합 (vLLM v1 스타일)
  ├─ ptr/vllm_multiseq/   Triton 커널 2개, split-dispatch (vLLM v0 스타일)
  └─ flashinfer/  (← 여기)  외부 라이브러리, split-dispatch (라이브러리 포맷이 그렇게 강제함)
```

**이 세 프로젝트를 나란히 두고 backend.py 를 diff 해 보면**:

1. **`get_kv_cache_shape` 완전 동일** — vLLM v1 의 KV-cache layout 은 backend 가 자유롭게 선택하는 것이 아니라, 세 backend 가 모두 같은 NHD 를 택한 결과 일치한 것뿐이다. (FlashInfer 는 NHD/HND 다 지원하지만 우리는 NHD 를 고르자마자 vllm_unified 와 호환되게 되었다.)

2. **Metadata 가 backend 특수** — vllm_unified 는 `block_table + seq_lens` 그대로, flashinfer 는 CSR 트리플 × 2그룹을 들고 있다. 이게 attention 라이브러리 교체 시의 **실제 논리적 비용**이다.

3. **Impl.forward 가 launch 패턴을 결정** — vllm_unified 는 1회 호출로 끝, flashinfer 는 plan/run 이 그룹별로 갈라진다. 이 차이가 프로덕션에서 TTFT/throughput 에 미치는 영향을 세뻀하게 볼 수 있다.

**파겴된 것 (이 프로젝트에서 단순화)**:
- `cudagraph_support = NEVER` — wrapper 의 `use_cuda_graph=True` 모드 미사용
- `MultiLevelCascadeAttentionWrapper` 미사용 — 공유 프리픽스 캡쉌 최적화 제외
- `fast_decode_plan` 미사용 — 매 forward `plan()` 재호출 (학습 가시성)
- CSR 변환을 Python 루프로 — 실제 vLLM 은 GPU 커널로 처리